# Paper-Aligned SSL-GGL Workflow

This notebook assembles the stored anchor observations, applies the paper's centered log-ratio (CLR) and Z-score transforms, rebuilds heterogeneous node features and Eq. 1–3 edge weights, trains the five Table 1 configurations, and runs the paper's post hoc interpretation analyses.

The data bundle uses compressed arrays, CSV tables, and JSON metadata. All coordinates are expressed in a local metric system after subtracting the minimum bundled node x and y; the original CRS and offsets are not stored. Source shapefiles and GeoTIFFs are not copied into this folder. The stored geometry-derived node locations, edge pairs, entity-member CSR arrays, and calibrated geological evidence are the input products; the notebook reconstructs the feature matrix and Eq. 1–3 edge weights. Geometry extraction and constraint calibration themselves cannot be regenerated without the omitted source vector/raster layers.

The paper's Table 1 experiment contains five configurations, ten positive-label fractions, and ten random seeds (500 fine-tuning runs and 20 pretraining runs). `TRAINING_MODE="fresh"` runs this paper-aligned design into a new output folder. The older seven-group, 700-run archive is retained only as provenance and is not used as a paper result because its edge weights, edge objective, and seed lineage differ from the manuscript-aligned workflow.

Evidence boundaries: the paper reports CLR followed by Z-score scaling, but does not specify zero replacement. All stored valid concentrations are strictly positive, so no replacement is needed. The archived run did not retain its original final fine-tuned weights; this notebook trains a new checkpoint under the paper-aligned graph weights and binary edge objective before post hoc analysis.

## 01 | Configure the workspace and load dependencies

Run Jupyter with this folder as the working directory. All inputs, support code, caches, and outputs are resolved relative to this folder.

In [ ]:
from pathlib import Path
import json
import sys
import time
import shutil
import math
import hashlib
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.stats import rankdata


_cwd = Path.cwd().resolve()
_package_candidates = [_cwd, *_cwd.parents]
PACKAGE_ROOT = next((candidate for candidate in _package_candidates if (candidate / "code" / "model").is_dir()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("Could not locate the project root. Start the notebook from this project or one of its child directories.")
_external_root = os.environ.get("SSL_GGL_DATA_ROOT")
ROOT = Path(_external_root).expanduser().resolve() if _external_root else PACKAGE_ROOT
if not (ROOT / "data" / "inputs").is_dir():
    raise FileNotFoundError("Input data are intentionally excluded from this package. Set SSL_GGL_DATA_ROOT to an external data root containing data/inputs/.")
DATA_DIR = ROOT / "data" / "inputs"
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
SUPPORT_DIR = PACKAGE_ROOT / "code" / "model"
REFERENCE_DIR = RESULTS_DIR / "interpretation_archive"

for directory in (PROCESSED_DIR, FIGURES_DIR, TABLES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "axes.grid": False})
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({"root": str(ROOT), "device": str(DEVICE), "cuda": torch.cuda.is_available()})

## 02 | Load and validate the compressed inputs

The chemistry matrix has one row per valid 1-km anchor cell and one column per element. The stored graph preserves node order, relation labels, pair distances, scale weights, and anonymized local coordinates; edge weights are recalculated below from manuscript Eqs. 1–3.

In [ ]:
chem_archive = np.load(DATA_DIR / "chemistry_concentrations.npz", allow_pickle=False)
geo_archive = np.load(DATA_DIR / "geophysics_observations.npz", allow_pickle=False)
graph_archive = np.load(DATA_DIR / "graph_structure.npz", allow_pickle=False)
labels_archive = np.load(DATA_DIR / "labels_targets_and_scores.npz", allow_pickle=False)
aggregation_archive = np.load(DATA_DIR / "entity_aggregation_csr.npz", allow_pickle=False)
target_membership_archive = np.load(DATA_DIR / "target_node_membership.npz", allow_pickle=False)
graph_metadata = json.loads((DATA_DIR / "graph_metadata.json").read_text(encoding="utf-8"))
model_config = json.loads((DATA_DIR / "model_config.json").read_text(encoding="utf-8"))
node_metadata = pd.read_csv(DATA_DIR / "node_metadata.csv")
constraints = pd.read_csv(DATA_DIR / "constraints.csv").sort_values("anchor_id").reset_index(drop=True)

chemistry_raw = chem_archive["concentrations"].astype(np.float64)
chem_names = chem_archive["feature_names"].astype(str).tolist()
geo_raw = geo_archive["values"].astype(np.float64)
geo_names = geo_archive["feature_names"].astype(str).tolist()
reference_node_features = graph_archive["reference_node_features"].astype(np.float32)
positive_mask = labels_archive["positive_mask"].astype(bool)
target_id = labels_archive["target_id"].astype(np.int16)
target_node_inside = target_membership_archive["node_inside"].astype(bool)
target_paper_ids = target_membership_archive["paper_target_ids"].astype(np.int16)
archived_selected_scores = labels_archive["selected_scores"].astype(np.float64)
anchor_xy = labels_archive["anchor_xy"].astype(np.float64)
anchor_grid_rc = labels_archive["anchor_grid_rc"].astype(np.int32)
n_anchors = len(positive_mask)
n_nodes = len(graph_archive["node_type"])

assert chemistry_raw.shape == (n_anchors, 39)
assert geo_raw.shape == (n_anchors, 2)
assert len(chem_names) == 39 and len(geo_names) == 2
assert n_anchors == 15550 and n_nodes == 16148
assert positive_mask.sum() == 36 and int((target_id > 0).sum()) == 556
assert target_node_inside.shape == (n_nodes, 4)
assert np.array_equal(target_paper_ids, np.arange(1, 5, dtype=np.int16))
assert np.array_equal(node_metadata["node_id"].to_numpy(), np.arange(n_nodes))
for target_column, paper_target_id in enumerate(target_paper_ids):
    assert np.array_equal(target_node_inside[:n_anchors, target_column], target_id == paper_target_id)
assert reference_node_features.shape == (n_nodes, 41)
assert len(constraints) == n_anchors
assert np.isfinite(chemistry_raw).all() and np.all(chemistry_raw > 0)
assert np.isfinite(geo_raw).all() and np.isfinite(anchor_xy).all()
assert np.array_equal(constraints.anchor_id.to_numpy(), np.arange(n_anchors))

pd.DataFrame([
    {"array": "chemistry_raw", "shape": str(chemistry_raw.shape), "dtype": str(chemistry_raw.dtype)},
    {"array": "geophysics_raw", "shape": str(geo_raw.shape), "dtype": str(geo_raw.dtype)},
    {"array": "graph nodes", "shape": str(graph_archive["node_type"].shape), "dtype": str(graph_archive["node_type"].dtype)},
    {"array": "directed edges", "shape": str(graph_archive["edge_index"].shape), "dtype": str(graph_archive["edge_index"].dtype)},
    {"array": "target polygon membership", "shape": str(target_node_inside.shape), "dtype": str(target_node_inside.dtype)},
]).set_index("array")

## 03 | Apply CLR and Z-score transformations

For each anchor, CLR subtracts the mean log concentration across the 39 geochemical channels. Each transformed channel is then standardized across all anchors with population standard deviation (`ddof=0`), matching the archived preprocessing parameters. Magnetic and gravity observations are standardized using the anchor-wide means and standard deviations stored in the graph metadata.

In [ ]:
log_concentrations = np.log(chemistry_raw)
clr_values = log_concentrations - log_concentrations.mean(axis=1, keepdims=True)
clr_means = clr_values.mean(axis=0)
clr_stds = clr_values.std(axis=0, ddof=0)
if np.any(clr_stds <= 0):
    raise ValueError("A geochemical CLR channel has zero variance")
chemistry_z = ((clr_values - clr_means) / clr_stds).astype(np.float32)

standardization = graph_metadata["input_standardization"]
geo_z_columns = []
geo_parameter_rows = []
for column, feature_name in enumerate(geo_names):
    mean = float(geo_raw[:, column].mean())
    std = float(geo_raw[:, column].std(ddof=0))
    archived = standardization[feature_name]
    if not np.isclose(mean, archived["anchor_mean"], rtol=1e-7, atol=1e-8):
        raise ValueError(f"Stored {feature_name} mean disagrees with the archived graph metadata")
    if not np.isclose(std, archived["anchor_std_ddof0"], rtol=1e-7, atol=1e-8):
        raise ValueError(f"Stored {feature_name} population SD disagrees with graph metadata")
    geo_z_columns.append((geo_raw[:, column] - mean) / std)
    geo_parameter_rows.append({"feature": feature_name, "mean": mean, "std_ddof0": std})
geophysics_z = np.column_stack(geo_z_columns).astype(np.float32)
anchor_features = np.column_stack([chemistry_z, geophysics_z]).astype(np.float32)

feature_names = chem_names + geo_names
pd.DataFrame({
    "feature": chem_names,
    "clr_mean_before_zscore": clr_means,
    "clr_std_before_zscore_ddof0": clr_stds,
}).to_csv(TABLES_DIR / "chemistry_clr_zscore_parameters.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(geo_parameter_rows).to_csv(TABLES_DIR / "geophysics_zscore_parameters.csv", index=False, encoding="utf-8-sig")
np.savez_compressed(
    PROCESSED_DIR / "anchor_features_41d.npz",
    features=anchor_features,
    feature_names=np.asarray(feature_names),
    chemistry_clr=clr_values.astype(np.float32),
)

anchor_feature_error = float(np.max(np.abs(anchor_features - reference_node_features[:n_anchors])))
if anchor_feature_error > 2e-5:
    raise ValueError(f"Reconstructed anchor features differ from the archived graph: {anchor_feature_error:.3g}")
print({"clr_formula": "log(x) - row_mean(log(x))", "zscore_ddof": 0, "max_anchor_feature_difference": anchor_feature_error})

## 04 | Assemble heterogeneous node features

Anchor nodes receive the reconstructed 41 input channels. Intrusion, carbonate, and contact nodes receive the mean feature vector of their stored aggregation members. The CSR membership arrays preserve the geometry-derived membership from the source graph; original vector geometries are not part of this bundle.

In [ ]:
node_features = np.zeros_like(reference_node_features, dtype=np.float32)
node_features[:n_anchors] = anchor_features
entity_node_ids = aggregation_archive["entity_node_ids"].astype(np.int64)
member_indptr = aggregation_archive["indptr"].astype(np.int64)
member_indices = aggregation_archive["indices"].astype(np.int64)
expected_counts = aggregation_archive["expected_counts"].astype(np.int64)

for row, node_id in enumerate(entity_node_ids):
    start, stop = member_indptr[row], member_indptr[row + 1]
    members = member_indices[start:stop]
    if len(members) != expected_counts[row] or len(members) == 0:
        raise ValueError(f"Invalid aggregation membership for node {node_id}")
    node_features[node_id] = anchor_features[members].mean(axis=0, dtype=np.float64)

if len(np.unique(entity_node_ids)) != len(entity_node_ids):
    raise ValueError("Entity aggregation node IDs are duplicated")
if len(entity_node_ids) != 598:
    raise ValueError(f"Expected 598 entity nodes, found {len(entity_node_ids)}")
graph_feature_error = float(np.max(np.abs(node_features - reference_node_features)))
if graph_feature_error > 2e-5:
    raise ValueError(f"Reconstructed heterogeneous node features differ from graph: {graph_feature_error:.3g}")

model_graph = {key: graph_archive[key] for key in graph_archive.files if key != "reference_node_features"}
model_graph["node_features"] = node_features
edge_distance = model_graph["edge_distance_m"].astype(np.float64)
scale_weight = model_graph["edge_scale_weight"].astype(np.float64)
edge_index = model_graph["edge_index"]
if np.any(edge_distance >= 1500.0) or not np.isfinite(edge_distance).all():
    raise ValueError("Stored graph contains an invalid edge distance for the paper's 1.5-km rule")
anchor_anchor = (edge_index[0] < n_anchors) & (edge_index[1] < n_anchors)
anchor_distances = np.linalg.norm(
    graph_archive["positions"][edge_index[0, anchor_anchor]].astype(np.float64)
    - graph_archive["positions"][edge_index[1, anchor_anchor]].astype(np.float64), axis=1
)
if np.max(np.abs(anchor_distances - edge_distance[anchor_anchor])) > 1e-3:
    raise ValueError("Stored anchor-anchor distances do not match projected node coordinates")
distance_min, distance_max = edge_distance.min(), edge_distance.max()
distance_affinity = (distance_max - edge_distance) / (distance_max - distance_min)
distance_weight = 0.2 + 0.8 * distance_affinity
weight_product = distance_weight * scale_weight
product_min, product_max = weight_product.min(), weight_product.max()
model_graph["edge_distance_affinity_raw"] = distance_affinity.astype(np.float32)
model_graph["edge_distance_weight"] = distance_weight.astype(np.float32)
model_graph["edge_weight_product"] = weight_product.astype(np.float32)
model_graph["edge_weight"] = (0.2 + 0.8 * (weight_product - product_min) / (product_max - product_min)).astype(np.float32)
if not np.isfinite(model_graph["edge_weight"]).all() or model_graph["edge_weight"].min() < 0.2 - 1e-6 or model_graph["edge_weight"].max() > 1.0 + 1e-6:
    raise ValueError("Eq. 1–3 edge weights must be finite and lie in [0.2, 1]")
graph_metadata["edge_weight_method"] = "manuscript Eqs. 1-3: linear distance and entity-scale weights, product min-max scaled to [0.2, 1]"
np.savez_compressed(PROCESSED_DIR / "model_input_arrays.npz", **model_graph)
print({"nodes": n_nodes, "edges": int(model_graph["edge_index"].shape[1]), "features": int(node_features.shape[1]), "max_graph_feature_difference": graph_feature_error, "edge_weight_range": [float(model_graph["edge_weight"].min()), float(model_graph["edge_weight"].max())], "distance_weight_formula": "0.2 + 0.8 * (d_max - d) / (d_max - d_min)"})

paper_feature_specs = [("Cu", "Cu"), ("Pb", "Pb"), ("Au", "Au"), ("U", "U"), ("Cr", "Cr"), ("Ni", "Ni"), ("Y", "Yt"), ("CaO", "CaO"), ("GRA", "Grav0_F"), ("MAG", "Mag0_F")]
input_dir = FIGURES_DIR / "inputs"
input_dir.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(2, 5, figsize=(15, 6), constrained_layout=True)
for ax, (label, source_name) in zip(axes.flat, paper_feature_specs):
    feature_id = feature_names.index(source_name)
    feature_grid = np.full(tuple(chem_archive["grid_shape"].astype(int)), np.nan, dtype=np.float32)
    feature_grid[anchor_grid_rc[:, 0], anchor_grid_rc[:, 1]] = node_features[:n_anchors, feature_id]
    image = ax.imshow(feature_grid, origin="upper", cmap="RdBu_r")
    ax.set_title(label)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(image, ax=ax, shrink=0.72)
fig.savefig(input_dir / "paper_feature_input_maps_2x5.png", bbox_inches="tight")
plt.close(fig)

## 05 | Build labels and geological constraint tensors

Known mineral occurrences are positive labels; the remaining anchors are unlabeled. The three evidence channels are retained separately for hard and soft geological constraints. The paper describes the geological evidence construction in Eqs. 20–23; the archived calibrated values are supplied as a compact array table.

In [ ]:
constraint_names = ["k_contact", "k_intrusion", "k_carbonate"]
constraint_values = constraints[constraint_names].to_numpy(dtype=np.float32)
if not np.isfinite(constraint_values).all() or np.any((constraint_values < 0) | (constraint_values > 1)):
    raise ValueError("Constraint evidence must be finite and lie in [0, 1]")
constraint_mean = constraint_values.mean(axis=1)
positive_ids = np.flatnonzero(positive_mask).astype(np.int64)
target_counts = {int(k): int((target_id == k).sum()) for k in range(1, 5)}
assert target_counts == {1: 293, 2: 33, 3: 116, 4: 114}

np.savez_compressed(
    PROCESSED_DIR / "labels_constraints_targets.npz",
    positive_mask=positive_mask,
    positive_ids=positive_ids,
    constraint_values=constraint_values,
    constraint_mean=constraint_mean.astype(np.float32),
    target_id=target_id,
    selected_scores=archived_selected_scores.astype(np.float32),
)
print({"known_positive_count": len(positive_ids), "unlabeled_anchor_count": n_anchors - len(positive_ids), "target_anchor_counts": target_counts})

## 06 | Load the model implementation and bind local arrays

The training implementation is bundled under `code/model`. The wrapper below supplies the external arrays directly, so training does not depend on historical absolute paths in the original experiment scripts.

In [ ]:
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))
import run_knowledge_injection_comparison as base
import run_experiment as experiment

base.DEVICE = DEVICE
experiment.DEVICE = DEVICE
base.GRAPH_SOURCE = PROCESSED_DIR / "model_input_arrays.npz"
base.GRAPH_METADATA_SOURCE = DATA_DIR / "graph_metadata.json"
experiment.PROJECT_ROOT = ROOT
experiment.EXPERIMENT_ROOT = RESULTS_DIR
experiment.OUTPUT_ROOT = RESULTS_DIR / "training_archive"
experiment.BASE_SEED = 20271111

training_data = {
    "graph": model_graph,
    "metadata": graph_metadata,
    "checkpoint": {"model_config": model_config},
    "finetune_config": {},
    "positive_ids": positive_ids,
    "positive_mask": positive_mask,
    "constraints": constraints,
}
base.load_data = lambda: training_data
print({"model_config": model_config, "training_device": str(experiment.DEVICE)})

## 07 | Verify tensorization and the complete training schedule

Table 1 defines five configurations: GCN, HGCN, GT, SSL-GT, and SSL-GGT. Each uses ten nested positive-label fractions and ten shared seeds, yielding 500 fine-tuning runs and 20 SSL pretraining runs. The code monitors the moving mean of training nnPU risk, not an independent validation score; all 36 known deposits are included in ranking metrics, matching the manuscript's reported in-sample evaluation scope.

In [ ]:
td_full = base.tensor_data(training_data)
td_full["config"] = dict(td_full["config"])
td_full["config"]["encoder_layers"] = 2
td_full["config"]["attention_query_block"] = experiment.QUERY_BLOCK
td_anchor = experiment.build_anchor_td(td_full)

paper_group_map = {
    "GCN_anchor_only": "GCN",
    "GCN_heterogeneous": "HGCN",
    "GCN-Transformer_direct": "GT",
    "GCN-Transformer_SSL": "SSL-GT",
    "GCN-Transformer_SSL_soft_hard": "SSL-GGT",
}
experiment.GROUPS = {name: experiment.GROUPS[name] for name in paper_group_map}

schedule = {
    "table_1_models": list(paper_group_map.values()),
    "positive_fractions_percent": list(experiment.FRACTIONS),
    "repeat_count": len(experiment.REPEATS),
    "fine_tuning_runs": len(experiment.GROUPS) * len(experiment.FRACTIONS) * len(experiment.REPEATS),
    "pretraining_runs": sum(config["pretrain"] for config in experiment.GROUPS.values()) * len(experiment.REPEATS),
    "max_pretraining_epochs": experiment.PRETRAIN_MAX_EPOCHS,
    "max_finetuning_epochs": experiment.FINETUNE_MAX_EPOCHS,
    "nnpu_positive_prior": 0.02,
    "early_stopping": {"minimum_epochs": experiment.MIN_EPOCHS, "patience": experiment.PATIENCE, "min_delta": experiment.MIN_DELTA},
    "heterogeneous_nodes": td_full["n_nodes"],
    "anchor_only_nodes": td_anchor["n_nodes"],
}
schedule

## 08 | Run the five-model paper comparison

The archived seven-group metrics do not match the manuscript-aligned edge weighting or binary pretraining loss. This cell therefore trains the five Table 1 models in a clean output directory. The runner saves all-label SSL-GGT checkpoints for the interpretation stage and can resume complete runs from this paper-specific folder.

In [ ]:
TRAINING_MODE = "fresh"
training_output = RESULTS_DIR / "training_paper_aligned"

experiment.OUTPUT_ROOT = training_output
experiment.BASE_SEED = 20271111
sys.argv = [str(SUPPORT_DIR / "run_experiment.py")]
if not torch.cuda.is_available():
    raise RuntimeError("The paper-aligned comparison runner requires CUDA")
experiment.main()

## 09 | Plot capture performance and cross-seed stability

The success-rate AUC and fixed-area capture summarize spatial ranking. Top-10% Jaccard measures overlap among the top-ranked anchor sets across repeats. These are descriptive resubstitution metrics because the known occurrences used for evaluation include sampled training positives.

In [ ]:
training_metrics = pd.read_csv(training_output / "run_metrics.csv")
expected_rows = len(experiment.GROUPS) * len(experiment.FRACTIONS) * len(experiment.REPEATS)
if len(training_metrics) != expected_rows or training_metrics.duplicated(["group", "fraction_percent", "repeat"]).any():
    raise ValueError(f"Training results are incomplete or duplicated: {len(training_metrics)} rows")

performance_dir = FIGURES_DIR / "performance"
performance_dir.mkdir(parents=True, exist_ok=True)
at_ten = training_metrics.loc[training_metrics.fraction_percent.eq(10)]
group_order = list(experiment.GROUPS)

fig, ax = plt.subplots(figsize=(11, 5))
box_values = [at_ten.loc[at_ten.group.eq(group), "capture_area_10_percent"].to_numpy() for group in group_order]
ax.boxplot(box_values, tick_labels=[paper_group_map[group] for group in group_order], showmeans=True)
ax.set_ylabel("Known deposits captured in top 10% area (%)")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(performance_dir / "capture_area_10_percent_by_model.png", bbox_inches="tight")
plt.show()
plt.close(fig)

capture_summary = training_metrics.groupby(["group", "fraction_percent"], as_index=False).agg(
    capture_mean=("capture_area_10_percent", "mean"), capture_sd=("capture_area_10_percent", "std"),
    success_auc_mean=("success_rate_curve_auc", "mean"), success_auc_sd=("success_rate_curve_auc", "std"),
)
capture_summary.to_csv(TABLES_DIR / "training_fraction_summary.csv", index=False, encoding="utf-8-sig")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for group in group_order:
    part = capture_summary.loc[capture_summary.group.eq(group)].sort_values("fraction_percent")
    label = paper_group_map[group]
    axes[0].errorbar(part.fraction_percent, part.capture_mean, yerr=part.capture_sd, marker="o", capsize=2, label=label)
    axes[1].errorbar(part.fraction_percent, part.success_auc_mean, yerr=part.success_auc_sd, marker="o", capsize=2, label=label)
axes[0].set(xlabel="Known-positive fraction used for training (%)", ylabel="Capture in top 10% area (%)")
axes[1].set(xlabel="Known-positive fraction used for training (%)", ylabel="Success-rate curve AUC")
axes[1].legend(fontsize=7, frameon=False)
fig.tight_layout()
fig.savefig(performance_dir / "capture_and_success_auc_by_fraction.png", bbox_inches="tight")
plt.show()
plt.close(fig)

def top_area_ids(scores: np.ndarray, fraction: float = 0.10) -> np.ndarray:
    count = int(np.ceil(len(scores) * fraction))
    return np.argsort(-scores, kind="mergesort")[:count]

stability_rows = []
rng = np.random.default_rng(20271117)
for group in group_order:
    for fraction in experiment.FRACTIONS:
        top_sets = []
        for repeat in experiment.REPEATS:
            score_path = training_output / "runs" / group / f"positive_{fraction:03d}" / f"repeat_{repeat:02d}" / "scores_and_labels.npz"
            with np.load(score_path, allow_pickle=False) as archive:
                top_sets.append(set(top_area_ids(archive["scores"].astype(np.float64))))
        pair_values = np.asarray([
            len(top_sets[i] & top_sets[j]) / len(top_sets[i] | top_sets[j])
            for i in range(len(top_sets)) for j in range(i + 1, len(top_sets))
        ])
        boot = rng.choice(pair_values, size=(2000, len(pair_values)), replace=True).mean(axis=1)
        stability_rows.append({
            "group": group, "paper_model": paper_group_map[group], "fraction_percent": fraction,
            "jaccard_mean": float(pair_values.mean()),
            "bootstrap_ci_low": float(np.quantile(boot, 0.025)),
            "bootstrap_ci_high": float(np.quantile(boot, 0.975)),
            "pair_count": int(len(pair_values)),
        })

stability = pd.DataFrame(stability_rows)
stability.to_csv(TABLES_DIR / "top10_jaccard_by_fraction.csv", index=False, encoding="utf-8-sig")
fig, ax = plt.subplots(figsize=(9, 5))
for group in group_order:
    part = stability.loc[stability.group.eq(group)].sort_values("fraction_percent")
    ax.plot(part.fraction_percent, part.jaccard_mean, marker="o", label=group)
    ax.fill_between(part.fraction_percent, part.bootstrap_ci_low, part.bootstrap_ci_high, alpha=0.12)
ax.set(xlabel="Known-positive fraction used for training (%)", ylabel="Pairwise Jaccard of top 10% anchors")
ax.legend(fontsize=7, frameon=False)
fig.tight_layout()
fig.savefig(performance_dir / "top10_jaccard_by_fraction.png", bbox_inches="tight")
plt.show()
plt.close(fig)

at_ten.loc[at_ten.group.eq("GCN-Transformer_SSL_soft_hard"), ["capture_area_10_percent", "success_rate_curve_auc"]].agg(["mean", "std"])

## 10 | Select the all-label SSL-GGT checkpoint

The best all-label SSL-GGT checkpoint is selected by descending 10%-area capture, then success-rate AUC, with repeat number as a deterministic tie-break. The notebook only selects weights produced by the paper-aligned training run.

In [ ]:
MODEL_GROUP = "GCN-Transformer_SSL_soft_hard"
model_output_dir = RESULTS_DIR / "final_model"
model_output_dir.mkdir(parents=True, exist_ok=True)

final_rows = training_metrics.loc[
    training_metrics.group.eq(MODEL_GROUP) & training_metrics.fraction_percent.eq(100)
].sort_values(["capture_area_10_percent", "success_rate_curve_auc", "repeat"], ascending=[False, False, True])
chosen = final_rows.iloc[0]
selected_repeat = int(chosen.repeat)
selected_checkpoint_path = training_output / "checkpoints" / MODEL_GROUP / "positive_100" / f"repeat_{selected_repeat:02d}" / "finetuned.pt"
checkpoint_source_note = "paper-aligned full-label SSL-GGT training run"

if not selected_checkpoint_path.is_file():
    raise FileNotFoundError(selected_checkpoint_path)
selected_checkpoint = torch.load(selected_checkpoint_path, map_location="cpu", weights_only=False)
selected_checkpoint_sha256 = hashlib.sha256(selected_checkpoint_path.read_bytes()).hexdigest()
shutil.copy2(selected_checkpoint_path, model_output_dir / "selected_finetuned.pt")
print({
    "group": MODEL_GROUP, "repeat": selected_repeat, "seed": int(selected_checkpoint["seed"]),
    "best_epoch": int(selected_checkpoint["best_epoch"]), "source": checkpoint_source_note,
    "sha256": selected_checkpoint_sha256,
})

## 11 | Score anchors and build the rank-quantile map

The map stores the within-study-area average-rank quantile `(rank - 1) / (n - 1)`. It is a relative ranking, not a calibrated probability. The output is saved as an NPY grid with NaN outside valid anchor cells.

In [ ]:
model = base.GraphProspectivityModel(model_config, experiment.GROUPS[MODEL_GROUP]["encoder"]).to(DEVICE)
model.load_state_dict(selected_checkpoint["state_dict"], strict=True)
experiment.prepare_gcn_cache(model)
model.eval()

with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
    model_outputs = experiment.model_forward(model, td_full, td_full["edge_index"], td_full["edge_weight"], hard=True)
    recomputed_scores = torch.sigmoid(model_outputs["prospectivity_logit"][:n_anchors].float()).cpu().numpy().astype(np.float64)

scores = recomputed_scores
rank_quantile = ((rankdata(scores, method="average") - 1) / max(len(scores) - 1, 1)).astype(np.float32)
grid_shape = tuple(chem_archive["grid_shape"].astype(int))
rank_grid = np.full(grid_shape, np.nan, dtype=np.float32)
rank_grid[anchor_grid_rc[:, 0], anchor_grid_rc[:, 1]] = rank_quantile
np.save(PROCESSED_DIR / "prospectivity_rank_quantile_grid.npy", rank_grid)
target_grid = np.zeros(grid_shape, dtype=np.int16)
target_grid[anchor_grid_rc[:, 0], anchor_grid_rc[:, 1]] = target_id
np.save(PROCESSED_DIR / "paper_target_id_grid.npy", target_grid)

rank_table = pd.DataFrame({
    "anchor_id": np.arange(n_anchors), "x": anchor_xy[:, 0], "y": anchor_xy[:, 1],
    "score": scores, "rank_quantile": rank_quantile,
    "is_known_positive": positive_mask, "paper_target_id": target_id,
})
rank_table.to_csv(TABLES_DIR / "anchor_prospectivity_scores.csv", index=False, encoding="utf-8-sig")

spatial_dir = FIGURES_DIR / "spatial"
spatial_dir.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(rank_grid, origin="upper", cmap="viridis", vmin=0, vmax=1)
for target in range(1, 5):
    ax.contour(target_grid == target, levels=[0.5], colors=["white"], linewidths=0.7)
    rows = anchor_grid_rc[target_id == target]
    ax.scatter(rows[:, 1], rows[:, 0], s=7, facecolors="none", edgecolors="white", linewidths=0.35, label=f"Target {target}")
ax.set(title="SSL-GGL anchor rank quantile", xlabel="Grid column", ylabel="Grid row")
ax.legend(frameon=False, fontsize=8, loc="upper right")
fig.colorbar(image, ax=ax, label="Average-rank quantile")
fig.tight_layout()
fig.savefig(spatial_dir / "ssl_ggl_rank_quantile.png", bbox_inches="tight")
plt.show()
plt.close(fig)

## 12 | Load and plot the retained legacy Figure 7 Integrated Gradients

Figure 7 uses the archived legacy per-anchor-logit Integrated Gradients for 556 anchors in the union of four target polygons, with 31 known deposits overlaid. It uses ten selected features and 32-step midpoint integration with centered finite differences (step 0.01), as documented in the archived method manifest. The complete legacy plotting table is loaded from `plot_data/figure_07/plot_points.csv`.


In [ ]:
# Load the retained legacy Figure 7 attribution table.
# The delivered Figure 7 covers 556 target-union anchors and 31 known deposits.
legacy_dir = Path("plot_data/figure_07")
if not legacy_dir.is_dir():
    legacy_dir = Path.cwd().parent / "plot_data" / "figure_07"
legacy_points = pd.read_csv(legacy_dir / "plot_points.csv")
legacy_selection = pd.read_csv(legacy_dir / "feature_selection.csv")
IG_FEATURE_LABELS = legacy_selection["feature"].tolist()
IG_FEATURE_NAMES = IG_FEATURE_LABELS.copy()
IG_STEPS = 32
IG_EPSILON = 0.01
legacy_anchor_ids = np.sort(legacy_points["anchor_id"].unique())
legacy_pivot = legacy_points.pivot(index="anchor_id", columns="feature", values="integrated_gradient")
ig_values = np.full((n_anchors, len(IG_FEATURE_LABELS)), np.nan, dtype=np.float32)
for j, feature in enumerate(IG_FEATURE_LABELS):
    ids = legacy_pivot.index.to_numpy(dtype=np.int64)
    ig_values[ids, j] = legacy_pivot[feature].to_numpy(dtype=np.float32)
ig_frame = pd.DataFrame(ig_values[legacy_anchor_ids], columns=IG_FEATURE_LABELS)
ig_frame.insert(0, "anchor_id", legacy_anchor_ids)
ig_frame.to_csv(TABLES_DIR / "legacy_per_anchor_integrated_gradients.csv", index=False, encoding="utf-8-sig")


## 13 | Load and reproduce the retained legacy Figure 7 dependence analysis

Figure 7 uses the stored legacy per-anchor-logit Integrated Gradients table for 556 anchors in the union of four target polygons and highlights 31 known deposits. It is not an all-anchor analysis.


In [ ]:
# Reproduce the retained legacy Figure 7 dependence plot from stored points.
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, (_, selected_row) in zip(axes.flat, legacy_selection.iterrows()):
    subset = legacy_points[legacy_points["feature"] == selected_row["feature"]]
    background = subset[~subset["known_deposit"]]
    deposits = subset[subset["known_deposit"]]
    ax.scatter(background["standardized_input"], background["integrated_gradient"], c="#8a969c", s=28, alpha=0.35, linewidths=0)
    ax.scatter(deposits["standardized_input"], deposits["integrated_gradient"], c="#e77d56", s=95, alpha=0.95, edgecolors="white", linewidths=1.0)
    ax.axhline(0, color="#555555", linewidth=0.8)
    ax.set(title=f"({selected_row['panel']}) {selected_row['feature']}", xlabel="Standardized input", ylabel="Integrated Gradient")
axes[0, 0].legend(["Target-union anchors (n=556)", "Known deposits (n=31)"], frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "figure_7_legacy_local_ig.tif", dpi=300, bbox_inches="tight")
plt.close(fig)


## 14 | Compute or load geological entity retention masks

For each target anchor, one scalar mask is optimized per intrusion, carbonate, and contact-zone entity node. Anchor features, graph edges, and model parameters remain fixed, as in Eqs. 25–26. The paper does not report optimizer settings; the reproducible code choices are 8 epochs per anchor, Adam at 0.01, sparsity weight 1.0, and entropy weight 0.1. Results are cached for later runs.

In [ ]:
RECOMPUTE_ENTITY_MASKS = False
mask_dir = RESULTS_DIR / "entity_masks"
mask_dir.mkdir(parents=True, exist_ok=True)
mask_path = mask_dir / "per_anchor_entity_masks.csv"
mask_manifest_path = mask_dir / "mask_manifest.json"
entity_types = [("intrusion", 1), ("carbonate", 2), ("contact", 3)]
node_type_ids = graph_metadata["node_type_ids"]
entity_node_ids_for_mask = np.concatenate([
    np.flatnonzero(model_graph["node_type"] == node_type_ids[name]) for name, _ in entity_types
]).astype(np.int64)

def model_logits_for_features(features):
    step_data = dict(td_full)
    step_data["features"] = features
    return experiment.model_forward(model, step_data, td_full["edge_index"], td_full["edge_weight"], hard=True)["prospectivity_logit"][:n_anchors].float()

def recompute_entity_masks(epochs=8, learning_rate=0.01, sparsity=1.0, entropy_weight=0.1):
    entity_ids_t = torch.as_tensor(entity_node_ids_for_mask, dtype=torch.long, device=DEVICE)
    target_ids_t = np.flatnonzero(target_id > 0).astype(np.int64)
    with torch.no_grad():
        base_logits = model_logits_for_features(td_full["features"]).detach()
    masks = np.full((len(target_ids_t), len(entity_node_ids_for_mask)), np.nan, dtype=np.float32)
    records = []
    for local_idx, anchor_id in enumerate(target_ids_t):
        mask_logits = torch.nn.Parameter(torch.zeros(len(entity_ids_t), device=DEVICE))
        optimizer = torch.optim.Adam([mask_logits], lr=learning_rate)
        for _ in range(epochs):
            optimizer.zero_grad(set_to_none=True)
            mask = torch.sigmoid(mask_logits)
            masked_features = td_full["features"].clone()
            masked_features[entity_ids_t] = masked_features[entity_ids_t] * mask[:, None]
            predicted = model_logits_for_features(masked_features)[int(anchor_id)]
            entropy = -(mask * torch.log(mask + 1e-15) + (1 - mask) * torch.log(1 - mask + 1e-15)).mean()
            loss = (predicted - base_logits[int(anchor_id)]).square() + sparsity * mask.mean() + entropy_weight * entropy
            loss.backward()
            optimizer.step()
        masks[local_idx] = torch.sigmoid(mask_logits).detach().cpu().numpy()
        if (local_idx + 1) % 25 == 0 or local_idx + 1 == len(target_ids_t):
            print(f"Entity masks {local_idx + 1}/{len(target_ids_t)}")
    for row, anchor_id in enumerate(target_ids_t):
        for col, entity_node_id in enumerate(entity_node_ids_for_mask):
            records.append({
                "anchor_id": int(anchor_id), "paper_target_id": int(target_id[anchor_id]),
                "entity_node_id": int(entity_node_id), "entity_mask": float(masks[row, col]),
            })
    mask_table = pd.DataFrame(records).merge(
        node_metadata[["node_id", "node_type", "local_id"]], left_on="entity_node_id", right_on="node_id", how="left"
    ).drop(columns="node_id")
    mask_table["selected_mask_gt_0_5"] = mask_table.entity_mask > 0.5
    mask_table.to_csv(mask_dir / "per_anchor_entity_masks.csv", index=False, encoding="utf-8-sig")
    return mask_table

mask_cache_is_current = False
if mask_path.is_file() and mask_manifest_path.is_file():
    mask_cache_manifest = json.loads(mask_manifest_path.read_text(encoding="utf-8"))
    mask_cache_is_current = mask_cache_manifest.get("checkpoint_sha256") == selected_checkpoint_sha256
if RECOMPUTE_ENTITY_MASKS or not mask_cache_is_current:
    entity_masks = recompute_entity_masks()
else:
    entity_masks = pd.read_csv(mask_path)
    if entity_masks.anchor_id.nunique() != 556 or entity_masks.entity_node_id.nunique() != 598:
        raise ValueError("Cached entity masks do not match this graph and target union")
mask_manifest_path.write_text(json.dumps({
    "checkpoint_sha256": selected_checkpoint_sha256,
    "model_group": MODEL_GROUP,
    "repeat": int(selected_repeat),
    "seed": int(selected_checkpoint["seed"]),
    "target_anchor_count": int(entity_masks.anchor_id.nunique()),
    "entity_node_count": int(entity_masks.entity_node_id.nunique()),
    "optimizer": "Adam",
    "epochs_per_anchor": 8,
    "learning_rate": 0.01,
    "sparsity_weight": 1.0,
    "entropy_weight": 0.1,
}, ensure_ascii=False, indent=2), encoding="utf-8")

entity_mask_summary = entity_masks.groupby(["paper_target_id", "node_type"], as_index=False).agg(
    anchor_count=("anchor_id", "nunique"), entity_count=("entity_node_id", "nunique"),
    mean_mask=("entity_mask", "mean"), median_mask=("entity_mask", "median"),
    selected_fraction=("selected_mask_gt_0_5", "mean"),
)
entity_mask_summary.to_csv(TABLES_DIR / "entity_mask_summary.csv", index=False, encoding="utf-8-sig")

## 15 | Plot Figure 8 high-retention entity contrasts

Entity nodes are ranked within each geological entity type by mean retention across the 556 target anchors. Figure 8 compares standardized neighboring-input values for the top-quartile entities and the remaining entities; these are not whole-rock assays.

In [ ]:
mean_entity_mask = entity_masks.groupby("entity_node_id").entity_mask.mean()
contrast_rows = []
for entity_name, type_id in entity_types:
    type_nodes = node_metadata.loc[node_metadata.node_type.eq(entity_name), "node_id"].to_numpy(dtype=np.int64)
    ranked_nodes = type_nodes[np.argsort(-mean_entity_mask.reindex(type_nodes).to_numpy(), kind="mergesort")]
    n_top = max(1, int(np.ceil(0.25 * len(ranked_nodes))))
    high_nodes, other_nodes = ranked_nodes[:n_top], ranked_nodes[n_top:]
    for j, feature in enumerate(feature_names):
        high_values = node_features[high_nodes, j]
        other_values = node_features[other_nodes, j]
        contrast_rows.append({
            "node_type": entity_name, "feature": feature,
            "top_quartile_entity_count": int(len(high_nodes)), "other_entity_count": int(len(other_nodes)),
            "top_quartile_median": float(np.median(high_values)),
            "other_median": float(np.median(other_values)),
            "median_difference": float(np.median(high_values) - np.median(other_values)),
        })
entity_contrasts = pd.DataFrame(contrast_rows)
entity_contrasts.to_csv(TABLES_DIR / "high_retention_entity_feature_contrasts.csv", index=False, encoding="utf-8-sig")

paper_entity_features = {
    "intrusion": ["Cu", "Ag", "Yt", "Ti", "SiO2"],
    "carbonate": ["Cu", "Ag", "Yt", "SiO2", "Nb"],
    "contact": ["Pb", "Ag", "Zn", "V", "Mn"],
}
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
for ax, (entity_name, _) in zip(axes, entity_types):
    type_nodes = node_metadata.loc[node_metadata.node_type.eq(entity_name), "node_id"].to_numpy(dtype=np.int64)
    ranked_nodes = type_nodes[np.argsort(-mean_entity_mask.reindex(type_nodes).to_numpy(), kind="mergesort")]
    n_top = max(1, int(np.ceil(0.25 * len(ranked_nodes))))
    high_nodes, other_nodes = ranked_nodes[:n_top], ranked_nodes[n_top:]
    features = paper_entity_features[entity_name]
    values = [node_features[group_nodes, feature_names.index(name)] for name in features for group_nodes in (high_nodes, other_nodes)]
    positions = np.arange(1, 2 * len(features) + 1)
    box = ax.boxplot(values, positions=positions, widths=0.65, patch_artist=True, showfliers=False)
    for index, patch in enumerate(box["boxes"]):
        patch.set_facecolor("#c4544f" if index % 2 == 0 else "#36818a")
        patch.set_alpha(0.72)
    ax.set_xticks(positions.reshape(-1, 2).mean(axis=1), features, rotation=35, ha="right")
    ax.set_title(entity_name.title())
    ax.set_ylabel("Standardized neighboring input")
    ax.grid(axis="y", alpha=0.16)
from matplotlib.patches import Patch
fig.legend(handles=[Patch(facecolor="#c4544f", alpha=0.72, label="High-retention top 25%"), Patch(facecolor="#36818a", alpha=0.72, label="Remaining entities")], loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(FIGURES_DIR / "figure_8_entity_input_contrasts.png", bbox_inches="tight")
plt.show()
plt.close(fig)

## 16 | Build Figure 9 target-specific panels

Each row is one paper target. The center panel follows the target-wise mask analysis: for each target anchor, it subtracts the mean mask of entity nodes outside the target polygon from the mean mask of entity nodes inside it, then shows the distribution across anchors. The reported 95% confidence interval is the mean difference plus or minus 1.96 standard errors. Polygon coverage is loaded from the compact, stored node-membership array, so entity centroids use their exact target membership rather than a nearest-anchor approximation. The right panel compares standardized anchor inputs inside and outside the target polygon.

In [ ]:
target_retention_rows = []
target_input_rows = []
fig, axes = plt.subplots(4, 3, figsize=(16, 16))
for target in range(1, 5):
    target_column = int(np.flatnonzero(target_paper_ids == target)[0])
    inside_node_mask = target_node_inside[:, target_column]
    inside_ids = np.flatnonzero(inside_node_mask[:n_anchors])
    outside_ids = np.flatnonzero(~inside_node_mask[:n_anchors])
    target_ig = ig_values[inside_ids]
    target_abs = np.mean(np.abs(target_ig), axis=0)
    target_signed = np.mean(target_ig, axis=0)
    top_idx = np.argsort(-target_abs, kind="mergesort")[:len(IG_FEATURE_LABELS)]
    y = np.arange(len(top_idx))
    axes[target - 1, 0].barh(y - 0.18, target_abs[top_idx], height=0.34, color="#36818a", label="Mean |IG|")
    axes[target - 1, 0].barh(y + 0.18, target_signed[top_idx], height=0.34, color=np.where(target_signed[top_idx] >= 0, "#c4544f", "#75838a"), label="Mean signed IG")
    axes[target - 1, 0].set_yticks(y, [IG_FEATURE_LABELS[i] for i in top_idx])
    axes[target - 1, 0].invert_yaxis()
    axes[target - 1, 0].axvline(0, color="#555555", linewidth=0.6)
    axes[target - 1, 0].set_title(f"Target {target}: IG")
    axes[target - 1, 0].set_xlabel("Attribution")
    if target == 1:
        axes[target - 1, 0].legend(frameon=False, fontsize=7)

    target_masks = entity_masks.loc[entity_masks.anchor_id.isin(inside_ids)]
    mask_differences = []
    for entity_name, _ in entity_types:
        type_nodes = node_metadata.loc[node_metadata.node_type.eq(entity_name), "node_id"].to_numpy(dtype=np.int64)
        type_inside_mask = inside_node_mask[type_nodes]
        inside_entity_nodes = type_nodes[type_inside_mask]
        outside_entity_nodes = type_nodes[~type_inside_mask]
        if not len(inside_entity_nodes) or not len(outside_entity_nodes):
            raise ValueError(f"Target {target} has no inside/outside {entity_name} nodes")
        type_masks = target_masks.loc[target_masks.node_type.eq(entity_name)]
        mask_by_anchor = type_masks.pivot(index="anchor_id", columns="entity_node_id", values="entity_mask")
        mask_by_anchor = mask_by_anchor.reindex(index=inside_ids, columns=type_nodes)
        inside_mean = mask_by_anchor.loc[:, inside_entity_nodes].mean(axis=1)
        outside_mean = mask_by_anchor.loc[:, outside_entity_nodes].mean(axis=1)
        delta = (inside_mean - outside_mean).dropna().to_numpy(dtype=np.float64)
        if len(delta) != len(inside_ids):
            raise ValueError(f"Target {target} has incomplete per-anchor {entity_name} masks")
        difference = float(delta.mean())
        sem = float(delta.std(ddof=1) / np.sqrt(len(delta))) if len(delta) > 1 else 0.0
        low, high = difference - 1.96 * sem, difference + 1.96 * sem
        mask_differences.append(delta)
        target_retention_rows.append({
            "target": target, "entity_type": entity_name, "target_anchor_count": int(len(delta)),
            "inside_entity_count": int(len(inside_entity_nodes)), "outside_entity_count": int(len(outside_entity_nodes)),
            "inside_mean": float(inside_mean.mean()), "outside_mean": float(outside_mean.mean()),
            "inside_minus_outside": difference,
            "ci_95_low": float(low), "ci_95_high": float(high),
        })
    mask_box = axes[target - 1, 1].boxplot(mask_differences, positions=np.arange(len(entity_types)), widths=0.5, patch_artist=True, showfliers=False)
    for patch in mask_box["boxes"]:
        patch.set_facecolor("#d5e0df")
        patch.set_edgecolor("#78918e")
    axes[target - 1, 1].set_xticks(np.arange(len(entity_types)), [name.title() for name, _ in entity_types], rotation=25, ha="right")
    axes[target - 1, 1].axhline(0, color="#555555", linewidth=0.6)
    axes[target - 1, 1].set_title(f"Target {target}: entity mask differences")
    axes[target - 1, 1].set_ylabel("Inside minus outside mean mask")

    input_diffs = np.median(node_features[inside_ids, :], axis=0) - np.median(node_features[outside_ids, :], axis=0)
    top_input_idx = np.argsort(-np.abs(input_diffs), kind="mergesort")[:10]
    plot_values = [values for j in top_input_idx for values in (node_features[inside_ids, j], node_features[outside_ids, j])]
    positions = np.arange(1, 2 * len(top_input_idx) + 1)
    box = axes[target - 1, 2].boxplot(plot_values, positions=positions, widths=0.65, patch_artist=True, showfliers=False)
    for index, patch in enumerate(box["boxes"]):
        patch.set_facecolor("#c4544f" if index % 2 == 0 else "#36818a")
        patch.set_alpha(0.72)
    target_input_features = [feature_names[j] for j in top_input_idx]
    axes[target - 1, 2].set_xticks(positions.reshape(-1, 2).mean(axis=1), target_input_features, rotation=35, ha="right")
    axes[target - 1, 2].set_title(f"Target {target}: anchor inputs")
    axes[target - 1, 2].set_ylabel("Standardized input")
    for j in top_input_idx:
        target_input_rows.append({
            "target": target, "feature": feature_names[j],
            "inside_median": float(np.median(node_features[inside_ids, j])),
            "outside_median": float(np.median(node_features[outside_ids, j])),
            "inside_minus_outside_median": float(input_diffs[j]),
            "inside_anchor_count": int(len(inside_ids)), "outside_anchor_count": int(len(outside_ids)),
        })

fig.legend(handles=[Patch(facecolor="#c4544f", alpha=0.72, label="Inside target"), Patch(facecolor="#36818a", alpha=0.72, label="Other targets")], loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.98))
fig.savefig(FIGURES_DIR / "figure_9_target_specific_interpretation.png", bbox_inches="tight")
plt.show()
plt.close(fig)
pd.DataFrame(target_retention_rows).to_csv(TABLES_DIR / "target_retention_inside_outside.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(target_input_rows).to_csv(TABLES_DIR / "target_input_inside_outside.csv", index=False, encoding="utf-8-sig")

## 17 | Export provenance and output index

The manifest records the selected checkpoint, Table 1 mapping, Eq. 1–3 graph weights, Eq. 15–17 pretraining objective, and interpretation scopes. `data_manifest.json` records the stored array dimensions and source hashes.

In [ ]:
run_manifest = {
    "paper": "Interpretability Analysis of Self-Supervised Heterogeneous Graph Deep Learning for Mineral Exploration Targeting",
    "manuscript_reference": "manuscript1118.docx (external reference; not required to execute this bundle)",
    "data_bundle": str(ROOT / "data_manifest.json"),
    "training_mode": TRAINING_MODE,
    "training_output": str(training_output),
    "training_run_count": int(len(training_metrics)),
    "table_1_model_mapping": paper_group_map,
    "selected_model": {
        "group": MODEL_GROUP, "repeat": int(selected_repeat),
        "seed": int(selected_checkpoint["seed"]), "source": checkpoint_source_note,
        "checkpoint_sha256": selected_checkpoint_sha256,
    },
    "chemistry_transform": "per-anchor CLR across 39 elements, then channel-wise Z-score over all anchors (ddof=0)",
    "geophysics_transform": "channel-wise Z-score using stored all-anchor means and population standard deviations",
    "graph": {
        "node_count": int(n_nodes), "edge_count": int(model_graph["edge_index"].shape[1]),
        "feature_count": int(node_features.shape[1]), "crs": graph_metadata["coordinate_crs"],
        "grid_transform": chem_archive["grid_transform"].astype(float).tolist(),
        "edge_rule_from_archived_metadata": graph_metadata["edge_rule"],
        "edge_weight_method": graph_metadata["edge_weight_method"],
    },
    "edge_pretraining_objective": "binary edge/non-edge prediction with BCEWithLogits; negatives sampled from non-edges with matching node-type combinations",
    "node_reconstruction_objective": "masked-channel MSE; the 25-of-41 masking ratio is a code choice because the paper does not report a ratio",
    "positive_prior_in_code": 0.02,
    "evaluation_note": "All 36 known deposits are included in ranking metrics; these are in-sample capture metrics, not spatial holdout validation.",
    "interpretation_note": "Integrated Gradients and entity masks describe model attribution/sensitivity, not geological causality.",
    "outputs": [
        "data/processed/model_input_arrays.npz",
        "results/tables/training_fraction_summary.csv",
        "results/tables/top10_jaccard_by_fraction.csv",
        "plot_data/figure_07/plot_points.csv",
        "results/tables/target_retention_inside_outside.csv",
        "results/tables/target_input_inside_outside.csv",
        "results/figures/performance",
        "results/figures/spatial",
        "figures/figure_07/figure_7_legacy_local_ig.tif",
        "results/figures/figure_8_entity_input_contrasts.png",
        "results/figures/figure_9_target_specific_interpretation.png",
    ],
}
(RESULTS_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
output_files = sorted(str(path.relative_to(ROOT)) for path in RESULTS_DIR.rglob("*") if path.is_file())
print({"output_file_count": len(output_files), "manifest": str(RESULTS_DIR / "run_manifest.json")})
pd.DataFrame({"output_file": output_files}).head(40)
